# 🌊 HCMC Flood Severity Prediction — Nhà Bè Tidal Flood Edition (v4)

**Focused Scope**: Huyện Nhà Bè & Nam Sài Gòn (Tuyến Rốn Ngập Triều Cường: Lê Văn Lương, Huỳnh Tấn Phát, Phạm Hữu Lầu, Nguyễn Hữu Thọ, Phước Kiển, Phú Xuân)

| Class | Trạng thái | Dấu hiệu vật lý | 🏍️ Xe máy | 🚗 Ô tô |
|:---:|---|---|---|---|
| **0** | 🟢 **Dry** | Đường khô ráo | ✅ An toàn | ✅ An toàn |
| **1** | 🔵 **Wet** | Ướt mặt đường / Đọng nước nông (<10cm) | ⚠️ Đi bình thường | ✅ An toàn |
| **2** | 🔴 **Flooded** | Triều cường dâng sâu (≥15cm, ngập pô xe) | ❌ KHÔNG ĐI | ⚠️ Cẩn thận |

**Core Pipeline**: Urban Street Flood Data + EfficientNet-B0 + Live Nhà Bè CCTV Domain Adaptation

In [ ]:
# ================================================================
# Cell 1 — Setup & Dependencies
# ================================================================
!pip install -q timm httpx kaggle 2>/dev/null

import os, glob, json, subprocess, shutil, random
import numpy as np
from pathlib import Path
from collections import Counter
from PIL import Image
import warnings
warnings.filterwarnings('ignore', message='.*Glyph.*')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import timm

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import httpx, urllib3
urllib3.disable_warnings()

# ── Google Drive Persistence ─────────────────────────────
BASE_DIR = '.'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/hcmc_flood_poc'
    print("✅ Google Drive mounted at", BASE_DIR)
except Exception:
    print("ℹ️  No Drive — saving to local workspace.")

DATA_DIR         = os.path.join(BASE_DIR, 'flood_data')
HCMC_SAMPLES_DIR = os.path.join(BASE_DIR, 'hcmc_samples')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(HCMC_SAMPLES_DIR, exist_ok=True)

# ── Hyperparameters ─────────────────────────────────────
BATCH_SIZE   = 32
IMG_SIZE     = 224
NUM_CLASSES  = 3
EPOCHS       = 20
LR           = 1e-4
PATIENCE     = 5

# ── v3: Confidence gate for Wet predictions on live CCTV ─
# If model says "Wet" but is less than this confident → treat as Dry
# Set to 0.0 to disable gating
WET_CONFIDENCE_GATE = 0.65

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES  = ['Dry', 'Wet', 'Flooded']
COLORS       = ['#2ecc71', '#3498db', '#e74c3c']

print("🖥️  Device:", DEVICE)
if DEVICE.type == 'cuda':
    print("   GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU not detected! Runtime > Change runtime type > T4 GPU")

## 1. 📥 Multi-Source Data Acquisition

Downloads 4 complementary street-level datasets:
1. **CVFD** (`vertexaisearch/close-view-flood-dataset-cvfd`): 7,855 images
2. **vjgopi Flood** (`vjgopi/flood-dataset`): ~2,000 images
3. **Mendeley Roadway** (Mendeley API fallback): 441 images + masks
4. **Flood Area Seg** (`faizalkarim/flood-area-segmentation`): 290 images + masks

**Kaggle Setup**: Click 🔑 Colab Secrets → add `KAGGLE_USERNAME` & `KAGGLE_KEY`.

In [ ]:
# ================================================================
# Cell 2 — Data Acquisition & Kaggle Authentication
# ================================================================
KAGGLE_READY = False

# Option 1: Colab Secrets (🔑 icon on sidebar)
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        KAGGLE_READY = True
        print("✅ Kaggle credentials loaded from Colab Secrets")
except Exception:
    pass

# Option 2: kaggle.json file in ~/.kaggle, notebook root, or Drive BASE_DIR
if not KAGGLE_READY:
    for kp in [os.path.expanduser('~/.kaggle/kaggle.json'), './kaggle.json', os.path.join(BASE_DIR, 'kaggle.json')]:
        if os.path.exists(kp):
            try:
                with open(kp) as f:
                    data = json.load(f)
                os.environ['KAGGLE_USERNAME'] = data.get('username', '')
                os.environ['KAGGLE_KEY']      = data.get('key', '')
                if os.environ['KAGGLE_USERNAME'] and os.environ['KAGGLE_KEY']:
                    KAGGLE_READY = True
                    print(f"✅ Kaggle credentials loaded from {kp}")
                    break
            except Exception:
                pass

# Option 3: Configured Kaggle credentials
if not KAGGLE_READY:
    KAGGLE_USER = "concacmemay"
    KAGGLE_KEY  = "ea6cbf4bcee92ccdc99678b3d6f7ce4c"
    if KAGGLE_USER and KAGGLE_KEY:
        os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
        os.environ['KAGGLE_KEY']      = KAGGLE_KEY
        KAGGLE_READY = True
        print(f"✅ Kaggle credentials set for user '{KAGGLE_USER}'")
    else:
        print("⚠️  Kaggle credentials not found.")

def dl_kaggle(slug, dest):
    if not KAGGLE_READY:
        return False
    os.makedirs(dest, exist_ok=True)
    r = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', slug, '--unzip', '-p', dest],
        capture_output=True, text=True
    )
    return r.returncode == 0

def dl_mendeley_direct(dest):
    os.makedirs(dest, exist_ok=True)
    existing = glob.glob(f"{dest}/**/image_*.jpg", recursive=True)
    if len(existing) > 100:
        print(f"  Already have {len(existing)} Mendeley images")
        return True
    print("  Fetching Mendeley file listing...")
    r = httpx.get('https://data.mendeley.com/public-api/datasets/t395bwcvbw', verify=False, timeout=30)
    files = r.json().get('files', [])
    client = httpx.Client(verify=False, follow_redirects=True, timeout=30)
    count = 0
    for f in tqdm(files, desc="  Downloading Mendeley"):
        fn = f['filename']
        if not (fn.endswith('.jpg') or fn.endswith('.png')):
            continue
        out = os.path.join(dest, fn)
        if os.path.exists(out):
            count += 1
            continue
        try:
            data = client.get(f['content_details']['download_url']).content
            with open(out, 'wb') as fp:
                fp.write(data)
            count += 1
        except Exception:
            pass
    client.close()
    print(f"  Got {count} files")
    return count > 0

print("=" * 60)
print("DOWNLOADING MULTI-SOURCE DATASETS")
print("=" * 60)

status = {}
DATASETS = [
    ('CVFD (7,855 imgs)',         'vertexaisearch/close-view-flood-dataset-cvfd', f"{DATA_DIR}/cvfd"),
    ('vjgopi Flood (2,000 imgs)', 'vjgopi/flood-dataset',                         f"{DATA_DIR}/vjgopi_flood"),
    ('Mendeley (441 imgs)',       'saurabhshahane/roadway-flooding-image-dataset', f"{DATA_DIR}/mendeley"),
    ('FloodSeg (290 imgs)',       'faizalkarim/flood-area-segmentation',           f"{DATA_DIR}/flood_seg"),
]

for name, slug, dest in DATASETS:
    print("📥 Downloading", name)
    if os.path.isdir(dest) and len(os.listdir(dest)) > 0:
        n_items = sum(1 for _ in Path(dest).rglob('*') if _.is_file())
        print("  Already exists", n_items, "files")
        status[name] = True
        continue
    if KAGGLE_READY:
        ok = dl_kaggle(slug, dest)
        status[name] = ok
        print("  Status:", "Done" if ok else "Failed")
    else:
        if 'Mendeley' in name:
            status[name] = dl_mendeley_direct(dest)
        else:
            status[name] = False
            print("  Skipped (requires Kaggle token)")

print("=" * 60)
for n, ok in status.items():
    print(f"  {n}: {'✅' if ok else '❌'}")
print("=" * 60)


## 2. 📊 Data Processing & Unified Labeling

**v3 change**: Tighter `mask_severity` thresholds.
- `thresh_dry = 0.15` (was 0.10) → images where ≤15% of pixels have mask coverage → Dry (reduces noise Wet labels from puddle-edge images)
- `thresh_sev = 0.40` (was 0.35) → requires more coverage to be Flooded, more is Wet

In [ ]:
# ================================================================
# Cell 3 — Data Processing & Robust Unified Labeling
# ================================================================
unified = []  # (image_path, severity_label)

def mask_severity(mask_path, thresh_dry=0.15, thresh_sev=0.40):
    try:
        m = np.array(Image.open(mask_path).convert('L'))
        ratio = (m > 30).mean()
        if ratio < thresh_dry:
            return 0  # Dry
        return 2 if ratio >= thresh_sev else 1  # 2: Flooded, 1: Wet
    except Exception:
        return None

# Helper: infer label from full path strings
def infer_label_from_path(fp):
    path_lower = fp.lower()
    # Check for Dry keywords first (non-flood, normal, dry, etc.)
    if any(k in path_lower for k in ['non-flood', 'non_flood', 'nonflooded', 'no_flood', 'not_flood', 'normal', 'dry', 'negative', 'clean']):
        return 0  # Dry
    # Check for Flood keywords
    if any(k in path_lower for k in ['flood', 'water', 'inundat', 'positive']):
        return 2  # Flooded
    return None

# ── 1. CVFD (7,855 images) ──────────────────────────────────
cvfd_dir = f"{DATA_DIR}/cvfd"
if os.path.isdir(cvfd_dir):
    print("Parsing CVFD...")
    n0 = len(unified)
    for root, dirs, files in os.walk(cvfd_dir):
        if any(k in root.lower() for k in ['mask', 'label', 'seg', 'annot']):
            continue
        for f in files:
            if not f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                continue
            fp = os.path.join(root, f)
            lbl = infer_label_from_path(fp)
            if lbl is None:
                # Default for unlabelled CVFD images
                lbl = 2 if 'flood' in f.lower() else 0
            unified.append((fp, lbl))
    print(f"  → Added {len(unified) - n0} CVFD samples")

# ── 2. vjgopi Flood Dataset (~2,000 images) ─────────────────
vj_dir = f"{DATA_DIR}/vjgopi_flood"
if os.path.isdir(vj_dir):
    print("Parsing vjgopi Flood Dataset...")
    n0 = len(unified)
    for root, dirs, files in os.walk(vj_dir):
        for f in files:
            if not f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                continue
            fp = os.path.join(root, f)
            lbl = infer_label_from_path(fp)
            if lbl is None:
                lbl = 2  # Default to flood for flood dataset
            unified.append((fp, lbl))
    print(f"  → Added {len(unified) - n0} vjgopi samples")

# ── 3. Mendeley (441 images + masks) ────────────────────────
mend_dir = f"{DATA_DIR}/mendeley"
if os.path.isdir(mend_dir):
    print("Parsing Mendeley...")
    n0 = len(unified)
    img_map, lbl_map = {}, {}
    for root, dirs, files in os.walk(mend_dir):
        for f in files:
            fp = os.path.join(root, f)
            if f.startswith('image_') and f.lower().endswith('.jpg'):
                img_map[f[6:-4]] = fp
            elif f.startswith('label_') and f.lower().endswith('.png'):
                lbl_map[f[6:-4]] = fp
    for idx, img_path in img_map.items():
        sev = mask_severity(lbl_map[idx]) if idx in lbl_map else None
        if sev is not None:
            unified.append((img_path, sev))
    print(f"  → Added {len(unified) - n0} Mendeley samples")

# ── 4. Flood Area Segmentation ───────────────────────────────
fseg_dir = f"{DATA_DIR}/flood_seg"
if os.path.isdir(fseg_dir):
    print("Parsing Flood Area Seg...")
    n0 = len(unified)
    img_dirs, mask_dirs = [], []
    for root, dirs, files in os.walk(fseg_dir):
        bn = os.path.basename(root).lower()
        if 'image' in bn and 'mask' not in bn:
            img_dirs.append(root)
        elif 'mask' in bn:
            mask_dirs.append(root)
    for img_d in img_dirs:
        for fp in glob.glob(f"{img_d}/*.*"):
            if not fp.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            name_ne = os.path.splitext(os.path.basename(fp))[0]
            mask_path = None
            for md in mask_dirs:
                for ext in ['.png', '.jpg']:
                    c = os.path.join(md, name_ne + ext)
                    if os.path.exists(c):
                        mask_path = c
                        break
            sev = mask_severity(mask_path) if mask_path else None
            if sev is not None:
                unified.append((fp, sev))
    print(f"  → Added {len(unified) - n0} FloodSeg samples")

# ── Verify images can be opened ─────────────────────────────
print("Raw collected samples:", len(unified))
valid = []
for path, label in tqdm(unified, desc="Verifying"):
    try:
        img = Image.open(path)
        img.verify()
        valid.append((path, label))
    except Exception:
        pass
unified = valid
print("Valid samples:", len(unified))

labels = [s for _, s in unified]
counts = Counter(labels)
assert counts.get(0, 0) > 0, f"[CRITICAL] Class 0 (Dry) is empty! Counts: {counts}"

# ── Class Distribution Plot ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
bars = axes[0].bar(CLASS_NAMES, [counts.get(i, 0) for i in range(NUM_CLASSES)],
                   color=COLORS, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + max(counts.values())*0.02,
                 f"{int(h):,}", ha='center', fontweight='bold', fontsize=12)
axes[0].spines[['top', 'right']].set_visible(False)
axes[1].pie([counts.get(i, 0) for i in range(NUM_CLASSES)],
            labels=CLASS_NAMES, colors=COLORS, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proportion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Sample Images Grid ───────────────────────────────────────
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(18, 4 * NUM_CLASSES))
for cls_idx in range(NUM_CLASSES):
    cls_samples = [p for p, l in unified if l == cls_idx]
    random.seed(42)
    random.shuffle(cls_samples)
    for j in range(5):
        ax = axes[cls_idx][j]
        if j < len(cls_samples):
            img = Image.open(cls_samples[j]).convert('RGB')
            ax.imshow(img)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(CLASS_NAMES[cls_idx], fontsize=14, fontweight='bold',
                          color=COLORS[cls_idx], rotation=0, labelpad=80, va='center')
plt.suptitle('Sample Images per Class (Dry / Wet / Flooded)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

for i, name in enumerate(CLASS_NAMES):
    c = counts.get(i, 0)
    print(f"  {name}: {c:,} ({100*c/len(unified):.1f}%)")


## 3. 🧠 EfficientNet-B0 Training Pipeline

**v3 augmentation additions** to fight false Wet predictions from CCTV:
- `GaussianBlur` — blurry low-res CCTV footage looks Dry
- `RandomErasing` — simulates shadow patches on road surface
- Wider `ColorJitter` saturation range — handles nighttime orange streetlight tint
- Increased contrast range — handles wet-looking glare from dry pavement

In [ ]:
# ================================================================
# Cell 4 — Dataset & DataLoaders
# ================================================================
class FloodDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            image = Image.open(path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        if self.transform:
            image = self.transform(image)
        return image, label

# ── v3 Train Transforms: Shadow/Blur/Reflection augmentation ─
train_tf = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    # Wider contrast + saturation: handles CCTV glare & night lighting
    T.ColorJitter(brightness=0.4, contrast=0.5, saturation=0.4, hue=0.08),
    # Simulate CCTV blur (low-res cameras)
    T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.5, 2.0))], p=0.4),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    # Simulate road shadow patches (rectangular dark blobs)
    # value=0 erases to mean color which is close to gray/shadow
    T.RandomErasing(p=0.3, scale=(0.02, 0.12), ratio=(0.2, 3.0), value=0),
])

val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

all_labels = [l for _, l in unified]
train_idx, temp_idx = train_test_split(
    range(len(unified)), test_size=0.3, stratify=all_labels, random_state=42)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5,
    stratify=[all_labels[i] for i in temp_idx], random_state=42)

train_set = FloodDataset([unified[i] for i in train_idx], train_tf)
val_set   = FloodDataset([unified[i] for i in val_idx],   val_tf)
test_set  = FloodDataset([unified[i] for i in test_idx],  val_tf)

train_labels = [unified[i][1] for i in train_idx]
cls_counts   = Counter(train_labels)
sample_weights = [1.0 / cls_counts[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

NW = 2 if DEVICE.type == 'cuda' else 0
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NW, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NW, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NW, pin_memory=True)

print(f"Train: {len(train_set):,}  |  Val: {len(val_set):,}  |  Test: {len(test_set):,}")

In [ ]:
# ================================================================
# Cell 5 — Model Definition & Training
# ================================================================
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

if len(unified) < 1000:
    print("⚠️  Small dataset (<1,000 samples). Freezing backbone.")
    for param in model.parameters():
        param.requires_grad = False
    if hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True
    elif hasattr(model, 'fc'):
        for param in model.fc.parameters():
            param.requires_grad = True

trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p     = sum(p.numel() for p in model.parameters())
print(f"Model params: {total_p:,}  |  Trainable: {trainable_p:,}")

# ── v3 Loss: Boost Dry class weight by ×2.5 ─────────────────
# Standard inverse-frequency weighting, but Dry gets an extra 2.5× multiplier
# This makes Dry→Wet misclassification 2.5× more costly in the loss
DRY_WEIGHT_BOOST = 2.5
base_w = torch.FloatTensor([1.0 / max(cls_counts.get(i, 1), 1) for i in range(NUM_CLASSES)])
base_w[0] *= DRY_WEIGHT_BOOST   # punish predicting Wet when truth is Dry
base_w = base_w / base_w.sum() * NUM_CLASSES
criterion = nn.CrossEntropyLoss(weight=base_w.to(DEVICE))
print(f"Loss weights → Dry: {base_w[0]:.3f}  Wet: {base_w[1]:.3f}  Flooded: {base_w[2]:.3f}")

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += out.argmax(1).eq(lbls).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    preds_all, labels_all = [], []
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss_sum += loss.item() * imgs.size(0)
        p = out.argmax(1)
        correct  += p.eq(lbls).sum().item()
        total    += imgs.size(0)
        preds_all.extend(p.cpu().numpy())
        labels_all.extend(lbls.cpu().numpy())
    return loss_sum / total, correct / total, preds_all, labels_all

print("🚀 Training...")
hist = {'tl': [], 'vl': [], 'ta': [], 'va': []}
best_vl, wait, best_state = float('inf'), 0, None

for epoch in range(EPOCHS):
    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()

    hist['tl'].append(tl); hist['vl'].append(vl)
    hist['ta'].append(ta); hist['va'].append(va)

    mark = ' ⬆️' if vl < best_vl else ''
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  Train {tl:.4f}/{ta:.3f}  Val {vl:.4f}/{va:.3f}{mark}")

    if vl < best_vl:
        best_vl, wait = vl, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"⏹️  Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_state)
model = model.to(DEVICE)
save_path = os.path.join(BASE_DIR, 'flood_model.pth')
torch.save(best_state, save_path)
print(f"✅ Done. Best val loss: {best_vl:.4f}")
print(f"💾 Model saved to {save_path}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(hist['tl']) + 1)
ax1.plot(ep, hist['tl'], 'b-o', ms=4, label='Train')
ax1.plot(ep, hist['vl'], 'r-o', ms=4, label='Val')
ax1.set_title('Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax1.spines[['top','right']].set_visible(False)
ax2.plot(ep, hist['ta'], 'b-o', ms=4, label='Train')
ax2.plot(ep, hist['va'], 'r-o', ms=4, label='Val')
ax2.set_title('Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 4. 📈 Comprehensive Evaluation

Pay attention to the **Dry row** in the confusion matrix. Any number in `Dry → Wet` is a false positive we want to minimize.

In [ ]:
# ================================================================
# Cell 6 — Evaluation
# ================================================================
VEHICLE_ADVICE = {
    0: {'🏍️ Xe máy': '✅ An toàn (Khô)', '🚗 Ô tô': '✅ An toàn'},
    1: {'🏍️ Xe máy': '⚠️ Đi bình thường (Ướt)', '🚗 Ô tô': '✅ An toàn'},
    2: {'🏍️ Xe máy': '❌ KHÔNG ĐI (Ngập ≥15cm)', '🚗 Ô tô': '⚠️ Cẩn thận'},
}

test_loss, test_acc, all_preds, all_true = evaluate(model, test_loader, criterion)
print(f"Test Accuracy: {test_acc:.4f}")
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES,
                             labels=range(NUM_CLASSES), digits=3, zero_division=0))

cm = confusion_matrix(all_true, all_preds, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
            annot_kws={'size': 14})
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix — v3 (Anti-False-Positive)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Dry→Wet false positive analysis ─────────────────────────
n_dry_total   = cm[0].sum()
n_dry_as_wet  = cm[0][1]
n_dry_as_flood= cm[0][2]
print("\n🔍 False Positive Analysis (Dry class):")
print(f"   True Dry total:       {n_dry_total}")
print(f"   Dry predicted as Wet: {n_dry_as_wet} ({100*n_dry_as_wet/max(n_dry_total,1):.1f}%)  ← TARGET < 10%")
print(f"   Dry pred as Flooded:  {n_dry_as_flood} ({100*n_dry_as_flood/max(n_dry_total,1):.1f}%)")

# ── Sample predictions grid ──────────────────────────────────
test_samples = [unified[i] for i in test_idx]
random.seed(123)
show_idx = random.sample(range(len(test_samples)), min(12, len(test_samples)))
rows = (len(show_idx) + 3) // 4
fig, axes = plt.subplots(rows, 4, figsize=(18, 5 * rows))
if rows == 1:
    axes = [axes]

for i, si in enumerate(show_idx):
    ax = axes[i // 4][i % 4]
    path, true_lbl = test_samples[si]
    img = Image.open(path).convert('RGB')
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    pred = probs.argmax().item()
    conf = probs[pred].item()

    ax.imshow(img.resize((IMG_SIZE, IMG_SIZE)))
    color = 'green' if pred == true_lbl else 'red'
    advice = VEHICLE_ADVICE[pred]
    adv_vals = list(advice.values())
    adv_str  = f'Moto: {adv_vals[0]} | Car: {adv_vals[1]}'
    title_str = "Pred: " + CLASS_NAMES[pred] + " (" + f"{conf:.0%}" + ")" + chr(10) + "True: " + CLASS_NAMES[true_lbl] + chr(10) + adv_str
    ax.set_title(title_str, color=color, fontsize=9)
    ax.axis('off')

for i in range(len(show_idx), rows * 4):
    axes[i // 4][i % 4].axis('off')

plt.suptitle('Predictions with Vehicle Advice', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 🏠 Live Nhà Bè & South Corridor CCTV Batch Diagnostic

Queries live CCTV cameras along core Nhà Bè tidal flood hotspots:
- **Lê Văn Lương** (Đường 15, Nguyễn Thị Thập, Long Kiểng, Rạch Đĩa)
- **Huỳnh Tấn Phát** (Phạm Hữu Lầu, Hoàng Quốc Việt, Phú Thuận, Trần Trọng Cung, Tân Thuận)
- **Nguyễn Hữu Thọ & Phạm Hữu Lầu** (Phước Kiển, Phú Xuân, Nhơn Đức)

In [ ]:
# ================================================================
# Cell 7 — Live HCMC Camera Fetch + Pre-Adaptation Diagnostic
# ================================================================
from io import BytesIO

print("🏙️  Loading HCMC camera database...")
cameras = []
try:
    r = httpx.get(
        'https://raw.githubusercontent.com/Thundercok/hcmc-flood-poc/master/cameras_list.json',
        verify=False, timeout=15)
    cameras = r.json()
    active = [c for c in cameras if c.get('CamStatus') == 'UP']
    print(f"Total cameras: {len(cameras):,}  |  Active: {len(active):,}")
except Exception as e:
    print(f"Could not load camera list: {e}")
    active = []

HOTSPOTS = [
    'nhà bè', 'nha be', 'lê văn lương', 'le van luong', 'huỳnh tấn phát', 'huynh tan phat',
    'phạm hữu lầu', 'pham huu lau', 'nguyễn hữu thọ', 'nguyen huu tho', 'phước kiển', 'phuoc kien',
    'nhơn đức', 'nhon duc', 'phú xuân', 'phu xuan', 'hiệp phước', 'hiep phuoc', 'phước lộc', 'phuoc loc',
    'nguyễn lương bằng', 'nguyen luong bang', 'hoàng quốc việt', 'hoang quoc viet',
    'trần trọng cung', 'tran trong cung', 'lưu trọng lư', 'luu trong lu', 'phú thuận', 'phu thuan'
]

def is_hotspot(cam):
    text = (cam.get('DisplayName') or '') + ' ' + (cam.get('Title') or '') + ' ' + (cam.get('Code') or '')
    text_lower = text.lower()
    for kw in HOTSPOTS:
        if kw in text_lower:
            return True, kw
    loc = cam.get('Location')
    if isinstance(loc, dict) and loc.get('__type') == 'DataTable':
        for row in loc.get('rows', []):
            loc_str = ' '.join([str(cell) for cell in row if cell]).lower()
            for kw in HOTSPOTS:
                if kw in loc_str:
                    return True, kw
    return False, None

hotspot_cams, other_cams = [], []
for cam in active:
    hit, kw = is_hotspot(cam)
    if hit:
        hotspot_cams.append(cam)
    else:
        other_cams.append(cam)

print(f"📍 Prioritized {len(hotspot_cams)} cameras in HCMC flood hotspot areas!")

# ── v3: confidence-gated inference ──────────────────────────
def predict_severity(image, gate=WET_CONFIDENCE_GATE):
    """Returns (raw_pred, gated_pred, probs).
    gated_pred: if raw_pred==Wet but confidence < gate → downgrade to Dry.
    """
    tensor = val_tf(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    raw_pred = int(probs.argmax())
    gated_pred = raw_pred
    if raw_pred == 1 and probs[1] < gate:  # Wet prediction below confidence gate
        gated_pred = 0  # Downgrade to Dry
    return raw_pred, gated_pred, probs

N_SAMPLE  = 60
sample_cams = (hotspot_cams + other_cams)[:N_SAMPLE] if active else []

print("=" * 60)
print(f"FETCHING {len(sample_cams)} HOTSPOT CAMERA SNAPSHOTS")
print("=" * 60)

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
client  = httpx.Client(verify=False, timeout=10, headers=headers)

results = []
os.makedirs(HCMC_SAMPLES_DIR, exist_ok=True)

for cam in sample_cams:
    cam_id = cam.get('CamId') or cam.get('Id')
    name   = cam.get('DisplayName', cam.get('Title', 'Camera'))
    url    = f'https://giaothong.hochiminhcity.gov.vn/render/ImageHandler.ashx?id={cam_id}'
    try:
        r = client.get(url)
        ctype = r.headers.get('content-type', '')
        if r.status_code == 200 and 'image' in ctype and len(r.content) > 1000:
            img = Image.open(BytesIO(r.content)).convert('RGB')
            raw_pred, gated_pred, probs = predict_severity(img)
            results.append({
                'name': name, 'cam_id': cam_id, 'img': img,
                'raw_pred': raw_pred, 'pred': gated_pred,
                'probs': probs, 'ok': True
            })
        else:
            results.append({'name': name, 'cam_id': cam_id, 'ok': False,
                            'reason': f'status={r.status_code} ctype={ctype}'})
    except Exception as err:
        results.append({'name': name, 'cam_id': cam_id, 'ok': False, 'reason': str(err)})

client.close()

ok_samples = [r for r in results if r['ok']]
print(f"✅ Got {len(ok_samples)} / {len(results)} camera snapshots")

# Save snapshots to persistent dir (used later for domain adaptation)
for i, r in enumerate(ok_samples):
    safe_name = r['name'].replace('/', '_').replace(' ', '_').replace(':', '_')[:40]
    out_file  = os.path.join(HCMC_SAMPLES_DIR, f"{i:02d}_{safe_name}.jpg")
    r['img'].save(out_file, quality=90)

print(f"📁 Saved {len(ok_samples)} snapshots to {HCMC_SAMPLES_DIR}")

if ok_samples:
    n_show = min(16, len(ok_samples))
    rows   = max((n_show + 3) // 4, 1)
    fig, axes = plt.subplots(rows, 4, figsize=(18, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    for i in range(n_show):
        r = ok_samples[i]
        pred_cls = r['pred']     # confidence-gated prediction
        raw_cls  = r['raw_pred']
        conf     = r['probs'][r['raw_pred']]
        advice   = VEHICLE_ADVICE[pred_cls]
        gated    = " [GATED→Dry]" if raw_cls != pred_cls else ""

        axes[i].imshow(r['img'].resize((IMG_SIZE, IMG_SIZE)))
        cam_title = r['name'][:22]
        cls_str   = CLASS_NAMES[pred_cls] + gated + " (" + f"{conf:.0%}" + ")"
        adv_vals  = list(advice.values())
        adv_str   = f'Moto: {adv_vals[0]} | Car: {adv_vals[1]}'
        full_title = cam_title + chr(10) + cls_str + chr(10) + adv_str
        axes[i].set_title(full_title, fontsize=9, fontweight='bold', color=COLORS[pred_cls])
        axes[i].axis('off')

    for i in range(n_show, len(axes)):
        axes[i].axis('off')

    plt.suptitle('HCMC Hotspot — v3 Predictions (Pre Domain-Adapt)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ── Before/After confidence gating stats ─────────────────
    raw_dist   = Counter(r['raw_pred'] for r in ok_samples)
    gated_dist = Counter(r['pred']     for r in ok_samples)
    gated_count = sum(1 for r in ok_samples if r['raw_pred'] != r['pred'])
    print("=" * 60)
    print("PRE-DOMAIN-ADAPTATION DIAGNOSTIC SUMMARY")
    print("=" * 60)
    print(f"Raw predictions:   {dict(raw_dist)}")
    print(f"After conf-gate:   {dict(gated_dist)}  ({gated_count} Wet→Dry downgrades)")
    confs = [r['probs'][r['pred']] for r in ok_samples]
    print(f"Confidence: Mean={np.mean(confs):.2%}, Min={np.min(confs):.2%}, Max={np.max(confs):.2%}")

## 6. 🧠 HCMC-Only Domain Adaptation

**v3 strategy** (stronger than v2):
- Fine-tunes **only the classifier head** (backbone frozen) on HCMC CCTV snapshots labeled as `Dry`
- Uses a **Dry-heavy synthetic dataset**: 1 real Dry snapshot → 5 augmented copies (ColorJitter + blur)
- Small LR (`5e-4`), only 8 epochs, early-stop patience 3
- **No mixing with academic datasets** — pure CCTV domain adaptation to prevent regressing on main data

In [ ]:
# ================================================================
# Cell 8 — HCMC-Only Domain Adaptation
# ================================================================
hcmc_dry_paths = sorted(glob.glob(os.path.join(HCMC_SAMPLES_DIR, '*.jpg')))
print(f"HCMC CCTV snapshots available for domain adaptation: {len(hcmc_dry_paths)}")

if len(hcmc_dry_paths) < 5:
    print("⚠️  Not enough HCMC snapshots (need ≥5). Skipping domain adaptation.")
    print("   → Run Cell 7 first to fetch camera snapshots.")
else:
    # ── Build HCMC-only fine-tune dataset ───────────────────
    # Use each CCTV Dry snapshot plus 4 random training Dry samples
    # to keep the model anchored — prevent catastrophic forgetting
    ANCHOR_DRY_N = min(200, counts.get(0, 0))  # anchor from academic Dry pool
    anchor_dry = [(p, 0) for p, l in unified if l == 0]
    random.seed(42)
    random.shuffle(anchor_dry)
    anchor_dry = anchor_dry[:ANCHOR_DRY_N]

    # HCMC snapshots: labeled as Dry (they are dry — no current flood)
    hcmc_samples = [(p, 0) for p in hcmc_dry_paths]

    # Combine: 5× overrepresent HCMC vs anchor (CCTV domain >> academic)
    adapt_data = anchor_dry + hcmc_samples * 5
    random.shuffle(adapt_data)

    adapt_labels = [l for _, l in adapt_data]
    adapt_counts = Counter(adapt_labels)
    print(f"Adapt dataset size: {len(adapt_data)}  Classes: {dict(adapt_counts)}")

    # ── CCTV-specific augmentation ──────────────────────────
    cctv_tf = T.Compose([
        T.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
        T.RandomCrop(IMG_SIZE),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
        T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.5, 2.5))], p=0.5),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.4, scale=(0.02, 0.15), ratio=(0.2, 4.0), value=0),
    ])

    # Train/val split on adapt data (80/20)
    split_n = int(0.8 * len(adapt_data))
    adapt_train = adapt_data[:split_n]
    adapt_val   = adapt_data[split_n:]

    adapt_train_set = FloodDataset(adapt_train, cctv_tf)
    adapt_val_set   = FloodDataset(adapt_val,   val_tf)

    adapt_train_labels = [l for _, l in adapt_train]
    adapt_cls_counts = Counter(adapt_train_labels)
    adapt_sw = [1.0 / adapt_cls_counts[l] for l in adapt_train_labels]
    adapt_sampler = WeightedRandomSampler(adapt_sw, len(adapt_sw), replacement=True)

    NW = 2 if DEVICE.type == 'cuda' else 0
    adapt_train_loader = DataLoader(adapt_train_set, batch_size=16, sampler=adapt_sampler,
                                    num_workers=NW, pin_memory=True)
    adapt_val_loader   = DataLoader(adapt_val_set,   batch_size=16, shuffle=False,
                                    num_workers=NW, pin_memory=True)

    # ── Freeze backbone, fine-tune head only ─────────────────
    for param in model.parameters():
        param.requires_grad = False
    if hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True
    elif hasattr(model, 'fc'):
        for param in model.fc.parameters():
            param.requires_grad = True

    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"🧠 Trainable parameters: {trainable_p:,} (backbone FROZEN)")

    # ── Loss: extra heavy on Dry to fight false positives ────
    adapt_w = torch.FloatTensor([4.0, 1.0, 1.0])  # Dry gets 4× weight
    criterion_adapt = nn.CrossEntropyLoss(weight=adapt_w.to(DEVICE))

    optimizer_adapt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=5e-4, weight_decay=1e-4)

    EPOCHS_ADAPT = 8
    best_vl2, wait2, best_state2 = float('inf'), 0, None

    print(f"🚀 Domain adaptation fine-tuning ({EPOCHS_ADAPT} epochs, Dry loss ×4.0)...")
    for epoch in range(EPOCHS_ADAPT):
        tl, ta = train_epoch(model, adapt_train_loader, criterion_adapt, optimizer_adapt)
        vl, va, _, _ = evaluate(model, adapt_val_loader, criterion_adapt)

        mark = ' ⬆️' if vl < best_vl2 else ''
        print(f"Epoch {epoch+1:2d}/{EPOCHS_ADAPT}  Train {tl:.4f}/{ta:.3f}  Val {vl:.4f}/{va:.3f}{mark}")

        if vl < best_vl2:
            best_vl2, wait2 = vl, 0
            best_state2 = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait2 += 1
            if wait2 >= 3:
                print(f"⏹️  Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state2)
    model = model.to(DEVICE)
    adapted_save_path = os.path.join(BASE_DIR, 'flood_model_v3_adapted.pth')
    torch.save(best_state2, adapted_save_path)
    print(f"✅ Adapted model saved: {adapted_save_path}  (Best val loss: {best_vl2:.4f})")

    # ── Re-run on original test set to check for regression ──
    print("\n🔁 Sanity-checking post-adapt test accuracy (should not regress badly)...")
    _, post_acc, post_preds, post_true = evaluate(model, test_loader, criterion)
    print(classification_report(post_true, post_preds, target_names=CLASS_NAMES,
                                 labels=range(NUM_CLASSES), digits=3, zero_division=0))

    cm2 = confusion_matrix(post_true, post_preds, labels=range(NUM_CLASSES))
    n_dry_total2   = cm2[0].sum()
    n_dry_as_wet2  = cm2[0][1]
    print(f"\nPost-adapt Dry→Wet FP: {n_dry_as_wet2}/{n_dry_total2} = {100*n_dry_as_wet2/max(n_dry_total2,1):.1f}%")

## 7. 🎯 Final Live Camera Evaluation (Post Domain Adaptation)

In [ ]:
# ================================================================
# Cell 9 — Post-Adaptation Live Camera Re-Evaluation
# ================================================================
if not ok_samples:
    print("No camera snapshots available. Re-run Cell 7.")
else:
    print("=" * 60)
    print("POST-DOMAIN-ADAPTATION: RE-PREDICTING ON LIVE CAMERAS")
    print("=" * 60)

    n_show = min(16, len(ok_samples))
    rows   = max((n_show + 3) // 4, 1)
    fig, axes = plt.subplots(rows, 4, figsize=(18, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    new_raw_preds, new_preds = [], []
    for i in range(n_show):
        r = ok_samples[i]
        raw_pred, gated_pred, probs = predict_severity(r['img'])
        new_raw_preds.append(raw_pred)
        new_preds.append(gated_pred)
        advice  = VEHICLE_ADVICE[gated_pred]
        gated   = " [GATED→Dry]" if raw_pred != gated_pred else ""

        axes[i].imshow(r['img'].resize((IMG_SIZE, IMG_SIZE)))
        cam_title = r['name'][:22]
        cls_str   = CLASS_NAMES[gated_pred] + gated + " (" + f"{probs[raw_pred]:.0%}" + ")"
        adv_vals  = list(advice.values())
        adv_str   = f'Moto: {adv_vals[0]} | Car: {adv_vals[1]}'
        full_title = cam_title + chr(10) + cls_str + chr(10) + adv_str
        axes[i].set_title(full_title, fontsize=9, fontweight='bold', color=COLORS[gated_pred])
        axes[i].axis('off')

    for i in range(n_show, len(axes)):
        axes[i].axis('off')

    plt.suptitle('POST-ADAPTATION: HCMC Hotspot Camera Predictions (v3)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    raw_dist_new   = Counter(new_raw_preds)
    gated_dist_new = Counter(new_preds)
    gated_count_new = sum(1 for rp, gp in zip(new_raw_preds, new_preds) if rp != gp)

    print("=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)
    print(f"Raw predictions:   {dict(raw_dist_new)}")
    print(f"After conf-gate:   {dict(gated_dist_new)}  ({gated_count_new} Wet→Dry downgrades)")
    confs_new = [r['probs'][r['pred']] if 'pred' in r else 0 for r in ok_samples[:n_show]]
    print(f"Confidence: Mean={np.mean(confs_new):.2%}, Min={np.min(confs_new):.2%}")
    print()
    # Count how many cameras are now Dry
    n_dry_post = gated_dist_new.get(0, 0)
    n_wet_post = gated_dist_new.get(1, 0)
    n_flood_post = gated_dist_new.get(2, 0)
    total_shown = n_show
    print(f"  🟢 Dry:     {n_dry_post}/{total_shown} cameras ({100*n_dry_post/total_shown:.0f}%)")
    print(f"  🔵 Wet:     {n_wet_post}/{total_shown} cameras ({100*n_wet_post/total_shown:.0f}%)")
    print(f"  🔴 Flooded: {n_flood_post}/{total_shown} cameras ({100*n_flood_post/total_shown:.0f}%)")